# TTA Comparison Analysis

Analyze results from `submit_tta_comparison.py` comparing TTA configurations.
Components: Align=Euclidean alignment, AdaBN=Adaptive BatchNorm (buffer size), CFT=Continual Fine-Tuning.

| Exp | Name | Align | AdaBN | Buffer | CFT |
|-----|------|-------|-------|--------|-----|
| 01 | baseline | none | ✗ | - | ✗ |
| 02 | adabn_noalign | none | ✓ | 32 | ✗ |
| 03 | adabn_align | euclidean | ✓ | 32 | ✗ |
| 04 | cft_noalign | none | ✗ | - | ✓ |
| 05 | cft_align | euclidean | ✗ | - | ✓ |
| 06 | full_new | euclidean | ✓ | 32 | ✓ |
| 07 | adabn_cft_noalign | none | ✓ | 32 | ✓ |
| 08 | adabn_old_align | euclidean | ✓ | 1 | ✗ |
| 09 | full_old | euclidean | ✓ | 1 | ✓ |

In [ ]:
# imports and config
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import re
from statsmodels.stats.multitest import multipletests
from scipy.stats import ttest_rel

# Per-subject balanced accuracy aggregated by generate_csv_revision.py
DATA_CSV = Path("figure_data/tta_comparison_subject_accuracy.csv")

# Map directory names to display names
# Dirs: 01_baseline, 02_adabn_noalign, etc.
METHOD_MAP = {
    "01_baseline": "Baseline",
    "02_adabn_noalign": "AdaBN(buf=32)",
    "08_adabn_old_align": "AdaBN(buf=1)+Align",
    "03_adabn_align": "AdaBN(buf=32)+Align",
    "04_cft_noalign": "CFT",
    "05_cft_align": "CFT+Align",
    "07_adabn_cft_noalign": "CFT+AdaBN(buf=32)",
    "09_full_old": "CFT+AdaBN(buf=1)+Align",
    "06_full_new": "CFT+AdaBN(buf=32)+Align",
}

# Non-CFT methods: blue hues, CFT methods: orange/red hues
METHOD_COLORS = {
    "Baseline": "#4393c3",
    "AdaBN(buf=32)": "#2166ac",
    "AdaBN(buf=32)+Align": "#053061",
    "CFT": "#fddbc7",
    "CFT+Align": "#f4a582",
    "CFT+AdaBN(buf=32)+Align": "#b2182b",
    "CFT+AdaBN(buf=32)": "#d6604d",
    "AdaBN(buf=1)+Align": "#92c5de",
    "CFT+AdaBN(buf=1)+Align": "#67001f",
}

MODEL_COLORS = {
    "ShallowConvNet": "#ca0020",
    "DeepConvNet": "#0571b0",
    "EEGNetv4": "#f4a582",
    "ATCNet": "#92c5de",
}


In [ ]:
# load per-subject balanced accuracy; one row per (method, dataset, model, fold, subject)
raw = pd.read_csv(DATA_CSV)
raw["method"] = raw["method"].map(METHOD_MAP)
raw = raw.dropna(subset=["method"])

# Filter to dataset-model pairs present in ALL methods
pairs_per_method = raw.groupby("method")[["dataset", "model"]].apply(
    lambda g: set(zip(g["dataset"], g["model"]))
)
common_pairs = set.intersection(*pairs_per_method.values)
print(f"Dataset-model pairs present in all {len(pairs_per_method)} methods: {len(common_pairs)}")
missing_pairs = set.union(*pairs_per_method.values) - common_pairs
if missing_pairs:
    print(f"Excluded (not in all methods): {sorted(missing_pairs)}")
raw = raw[[(d, m) in common_pairs for d, m in zip(raw["dataset"], raw["model"])]]

# a subject appears once per method/dataset/model: keep its lowest-fold entry
final_acc = (
    raw.sort_values("fold", kind="stable")
    .groupby(["method", "dataset", "model", "subject_id"], sort=True)
    .agg(balanced_acc=("balanced_acc", "first"))
    .reset_index()
)
print(f"Datasets: {sorted(final_acc['dataset'].unique())}")
print(f"Models: {sorted(final_acc['model'].unique())}")
print(f"Methods: {sorted(final_acc['method'].unique())}")


In [ ]:
# summary statistics of per-subject balanced accuracy
# summary stats aggregated across all subjects
summary = (
    final_acc.groupby(["method", "dataset", "model"])
    .agg(
        mean_acc=("balanced_acc", "mean"),
        std_acc=("balanced_acc", "std"),
        n_subjects=("subject_id", "nunique"),
    )
    .round(3)
)

# overall summary per method
overall_summary = (
    final_acc.groupby("method")
    .agg(
        mean_acc=("balanced_acc", "mean"),
        std_acc=("balanced_acc", "std"),
        n_subjects=("subject_id", "nunique"),
    )
    .round(3)
)

print("Overall balanced accuracy by method:")
print(overall_summary.sort_values("mean_acc", ascending=False))

In [ ]:
# bar plot: overall accuracy per method with significance vs baseline
method_order = list(METHOD_MAP.values())
overall_ordered = overall_summary.reindex(method_order)

# sort by mean accuracy in ascending order
overall_ordered = overall_ordered.sort_values("mean_acc")
method_order = overall_ordered.index.tolist()

# two-sided paired t-test: each method vs baseline (paired by dataset/model/subject)
baseline_method = "Baseline"
pvalues = {}

for method in method_order:
    if method == baseline_method:
        pvalues[method] = np.nan
        continue

    # get paired data: each row is (dataset, model, subject_id)
    baseline_acc = final_acc[final_acc["method"] == baseline_method].set_index(
        ["dataset", "model", "subject_id"]
    )["balanced_acc"]
    method_acc = final_acc[final_acc["method"] == method].set_index(
        ["dataset", "model", "subject_id"]
    )["balanced_acc"]

    # align indices
    common_idx = baseline_acc.index.intersection(method_acc.index)
    baseline_vals = baseline_acc.loc[common_idx].values
    method_vals = method_acc.loc[common_idx].values

    # two-sided paired t-test
    _, p_two = ttest_rel(method_vals, baseline_vals)
    pvalues[method] = p_two

# BH correction for multiple comparisons
methods_to_correct = [m for m in method_order if m != baseline_method]
p_raw = [pvalues[m] for m in methods_to_correct]
_, p_corrected, _, _ = multipletests(p_raw, method="fdr_bh")
for m, p_corr in zip(methods_to_correct, p_corrected):
    pvalues[m] = p_corr


# significance stars
def get_stars(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


fig, ax = plt.subplots(figsize=(6, 4))

bars = ax.bar(
    range(len(method_order)),
    overall_ordered["mean_acc"],
    # yerr=overall_ordered["std_acc"],
    color=[METHOD_COLORS[m] for m in method_order],
    edgecolor="black",
    capsize=3,
)

ax.set_xticks(range(len(method_order)))
ax.set_xticklabels(method_order, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Mean Bal. Acc.")
ax.set_title(f"Overall Bal. Acc. (vs {baseline_method})")
ax.set_ylim(0.5, 1)
ax.grid(True, alpha=0.3, axis="y")

for bar, method in zip(bars, method_order):
    val = overall_ordered.loc[method, "mean_acc"]
    if hasattr(val, "__iter__") and not isinstance(val, str):
        val = val.iloc[0] if len(val) > 0 else val
    stars = get_stars(pvalues[method])
    label = f"{val:.3f}{stars}"
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        label,
        ha="center",
        va="bottom",
        fontsize=8,
    )

plt.tight_layout()
plt.savefig("tta_comparison_overall_accuracy_barplot.pdf", bbox_inches="tight")
plt.show()
print(f"p-values (BH corrected, two-sided vs {baseline_method}):")
for m in method_order:
    print(f"  {m}: {pvalues[m]:.4f}" if not pd.isna(pvalues[m]) else f"  {m}: baseline")

In [ ]:
# heatmap: accuracy difference from baseline per dataset/model with significance
baseline_method = "Baseline"

# compute mean accuracy per method/dataset/model
pivot_data = summary.reset_index().pivot_table(
    index=["dataset", "model"], columns="method", values="mean_acc"
)

# compute difference from baseline
if baseline_method in pivot_data.columns:
    for col in pivot_data.columns:
        if col != baseline_method:
            pivot_data[f"{col} (Δ)"] = pivot_data[col] - pivot_data[baseline_method]

    # select only delta columns
    delta_cols = [c for c in pivot_data.columns if "(Δ)" in c]
    if delta_cols:
        delta_data = pivot_data[delta_cols]
        delta_data.columns = [c.replace(" (Δ)", "") for c in delta_data.columns]

        # sort columns by METHOD_MAP order (same as barplot)
        method_order = [m for m in METHOD_MAP.values() if m in delta_data.columns]
        print(method_order)
        # further sort by if CFT is there
        method_order = sorted(method_order, key=lambda x: "CFT" in x)
        delta_data = delta_data[method_order]

        # compute p-values for each cell (paired t-test per dataset/model)
        pval_data = pd.DataFrame(index=delta_data.index, columns=delta_data.columns)
        all_pvals = []  # for BH correction
        pval_coords = []  # (row_idx, col_idx)

        for i, (dataset, model) in enumerate(delta_data.index):
            baseline_subj = final_acc[
                (final_acc["method"] == baseline_method)
                & (final_acc["dataset"] == dataset)
                & (final_acc["model"] == model)
            ].set_index("subject_id")["balanced_acc"]

            for j, method in enumerate(delta_data.columns):
                if method == baseline_method:
                    continue

                method_subj = final_acc[
                    (final_acc["method"] == method)
                    & (final_acc["dataset"] == dataset)
                    & (final_acc["model"] == model)
                ].set_index("subject_id")["balanced_acc"]

                common_subj = baseline_subj.index.intersection(method_subj.index)
                if len(common_subj) < 2:
                    continue

                baseline_vals = baseline_subj.loc[common_subj].values
                method_vals = method_subj.loc[common_subj].values

                # two-sided paired t-test
                _, p_two = ttest_rel(method_vals, baseline_vals)
                all_pvals.append(p_two)
                pval_coords.append((i, j))

        # BH correction across all tests
        if all_pvals:
            _, p_corrected, _, _ = multipletests(all_pvals, method="fdr_bh")
            for (i, j), p_corr in zip(pval_coords, p_corrected):
                pval_data.iloc[i, j] = p_corr

        # significance stars function
        def get_stars(p):
            if pd.isna(p):
                return ""
            if p < 0.001:
                return "***"
            if p < 0.01:
                return "**"
            if p < 0.05:
                return "*"
            return ""

        fig, ax = plt.subplots(figsize=(8, max(3, len(delta_data) * 0.4)))

        im = ax.imshow(
            delta_data.values,
            cmap="RdYlGn",
            aspect="auto",
            vmin=-0.1,
            vmax=0.1,
            origin="lower",
        )

        ax.set_xticks(range(len(delta_data.columns)))
        ax.set_xticklabels(delta_data.columns, rotation=30, ha="right", fontsize=9)
        ax.set_yticks(range(len(delta_data.index)))
        ax.set_yticklabels([f"{d}/{m}" for d, m in delta_data.index], fontsize=9)

        # add values with significance stars
        for i in range(len(delta_data.index)):
            for j in range(len(delta_data.columns)):
                val = delta_data.iloc[i, j]
                p = pval_data.iloc[i, j]
                stars = get_stars(p)
                color = "white" if abs(val) > 0.05 else "black"
                ax.text(
                    j,
                    i,
                    f"{val:+.3f}{stars}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color=color,
                )

        plt.colorbar(im, ax=ax, label=f"Δ Accuracy vs {baseline_method}")
        ax.set_title(
            f"Accuracy change vs {baseline_method} (*p<.05, **p<.01, ***p<.001, BH-corrected)"
        )
        plt.tight_layout()
        plt.savefig("accuracy_heatmap_tta_ablations.pdf", bbox_inches="tight")
        plt.show()
else:
    print(f"{baseline_method} baseline not found in data")

In [ ]:
# %% 2-panel figure: barplot + heatmap side by side (full-width, fig2 sizing)
import string
import matplotlib.gridspec as gridspec

FONTSIZE = 7

# # cleaner TTA method labels for plotting
# TTA_LABELS = {
#     "Baseline": "Baseline",
#     "AdaBN": "AdaBN",
#     "AdaBN+Align": "AdaBN\n+Align",
#     "CFT": "CFT",
#     "CFT+Align": "CFT\n+Align",
#     "Full (buf=32)": "Full\n(buf=32)",
#     "AdaBN+CFT": "AdaBN\n+CFT",
#     "AdaBN+Align (buf=1)": "AdaBN+Align\n(buf=1)",
#     "Full (buf=1)": "Full\n(buf=1)",
# }
TTA_LABELS = {
    "Baseline": "Baseline",
    "AdaBN(buf=32)": "AdaBN(buf=32)",
    "AdaBN(buf=1)+Align": "AdaBN(buf=1)+Align",
    "AdaBN(buf=32)+Align": "AdaBN(buf=32)+Align",
    "CFT": "CFT",
    "CFT+Align": "CFT+Align",
    "CFT+AdaBN(buf=32)": "CFT+AdaBN(buf=32)",
    "CFT+AdaBN(buf=1)+Align": "CFT+AdaBN(buf=1)\n+Align",
    "CFT+AdaBN(buf=32)+Align": "CFT+AdaBN(buf=32)\n+Align",
}
# figure sizing (full-width)
fig = plt.figure(figsize=(7, 3.5))
gs = gridspec.GridSpec(1, 2, width_ratios=[1, 2], wspace=0.4)
panel_labels = list(string.ascii_lowercase)

# === Panel A: horizontal barplot ===
ax = fig.add_subplot(gs[0])
ax.text(
    -0.45,
    1.05,
    panel_labels[0],
    transform=ax.transAxes,
    fontsize=FONTSIZE + 2,
    fontweight="bold",
    va="top",
    ha="left",
)

method_order = list(METHOD_MAP.values())
overall_ordered = overall_summary.reindex(method_order)
overall_ordered = overall_ordered.sort_values("mean_acc")
method_order = overall_ordered.index.tolist()

# compute p-values vs baseline
baseline_method = "Baseline"
pvalues = {}
for method in method_order:
    if method == baseline_method:
        pvalues[method] = np.nan
        continue
    baseline_acc = final_acc[final_acc["method"] == baseline_method].set_index(
        ["dataset", "model", "subject_id"]
    )["balanced_acc"]
    method_acc = final_acc[final_acc["method"] == method].set_index(
        ["dataset", "model", "subject_id"]
    )["balanced_acc"]
    common_idx = baseline_acc.index.intersection(method_acc.index)
    baseline_vals = baseline_acc.loc[common_idx].values
    method_vals = method_acc.loc[common_idx].values
    _, p_two = ttest_rel(method_vals, baseline_vals)
    pvalues[method] = p_two

methods_to_correct = [m for m in method_order if m != baseline_method]
p_raw = [pvalues[m] for m in methods_to_correct]
_, p_corrected, _, _ = multipletests(p_raw, method="fdr_bh")
for m, p_corr in zip(methods_to_correct, p_corrected):
    pvalues[m] = p_corr


def get_stars(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


bars = ax.barh(
    range(len(method_order)),
    overall_ordered["mean_acc"],
    color=[METHOD_COLORS[m] for m in method_order],
    edgecolor="black",
)
ax.set_yticks(range(len(method_order)))
ax.set_yticklabels(
    [TTA_LABELS.get(m, m) for m in method_order],
    fontsize=FONTSIZE,
)
ax.set_xlabel("Mean Bal. Acc.", fontsize=FONTSIZE)
ax.set_xlim(0.7, 0.9)
ax.tick_params(axis="both", labelsize=FONTSIZE)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for bar, method in zip(bars, method_order):
    val = overall_ordered.loc[method, "mean_acc"]
    if hasattr(val, "__iter__") and not isinstance(val, str):
        val = val.iloc[0] if len(val) > 0 else val
    stars = get_stars(pvalues[method])
    # Place text inside the bar
    bar_width = bar.get_width()
    # Choose text color based on bar color brightness
    bar_color = METHOD_COLORS[method]
    # Convert color to RGB if it's a string
    import matplotlib.colors as mcolors

    rgb = mcolors.to_rgb(bar_color)
    # Calculate luminance to determine text color
    luminance = 0.299 * rgb[0] + 0.587 * rgb[1] + 0.114 * rgb[2]
    text_color = "white" if luminance < 0.5 else "black"
    ax.text(
        bar.get_width() - 0.005,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.2f}{stars}",
        ha="right",
        va="center",
        fontsize=FONTSIZE - 1,
        color=text_color,
    )
ax.set_title(
    "Mean Bal. Acc. across 3 datasets & 4 models", fontsize=FONTSIZE, weight="bold"
)

# === Panel B: heatmap ===
ax = fig.add_subplot(gs[1])
ax.text(
    -0.1,
    1.05,
    panel_labels[1],
    transform=ax.transAxes,
    fontsize=FONTSIZE + 2,
    fontweight="bold",
    va="top",
    ha="left",
)

pivot_data = summary.reset_index().pivot_table(
    index=["dataset", "model"], columns="method", values="mean_acc"
)

for col in pivot_data.columns:
    if col != baseline_method:
        pivot_data[f"{col} (Δ)"] = pivot_data[col] - pivot_data[baseline_method]

delta_cols = [c for c in pivot_data.columns if "(Δ)" in c]
delta_data = pivot_data[delta_cols]
delta_data.columns = [c.replace(" (Δ)", "") for c in delta_data.columns]

# Get all methods that exist in delta_data columns (not just those in TTA_LABELS)
available_methods = delta_data.columns.tolist()
# Order by TTA_LABELS first, then any remaining methods
method_order_heat = [m for m in TTA_LABELS.keys() if m in available_methods]
method_order_heat += [m for m in available_methods if m not in method_order_heat]
delta_data = delta_data[method_order_heat]

# compute p-values for heatmap
pval_data = pd.DataFrame(index=delta_data.index, columns=delta_data.columns)
all_pvals = []
pval_coords = []

for i, (dataset, model) in enumerate(delta_data.index):
    baseline_subj = final_acc[
        (final_acc["method"] == baseline_method)
        & (final_acc["dataset"] == dataset)
        & (final_acc["model"] == model)
    ].set_index("subject_id")["balanced_acc"]

    for j, method in enumerate(delta_data.columns):
        if method == baseline_method:
            continue
        method_subj = final_acc[
            (final_acc["method"] == method)
            & (final_acc["dataset"] == dataset)
            & (final_acc["model"] == model)
        ].set_index("subject_id")["balanced_acc"]
        common_subj = baseline_subj.index.intersection(method_subj.index)
        if len(common_subj) < 2:
            continue
        baseline_vals = baseline_subj.loc[common_subj].values
        method_vals = method_subj.loc[common_subj].values
        _, p_two = ttest_rel(method_vals, baseline_vals)
        all_pvals.append(p_two)
        pval_coords.append((i, j))

if all_pvals:
    _, p_corrected, _, _ = multipletests(all_pvals, method="fdr_bh")
    for (i, j), p_corr in zip(pval_coords, p_corrected):
        pval_data.iloc[i, j] = p_corr

im = ax.imshow(delta_data.values, cmap="RdYlGn", aspect="auto", vmin=-0.15, vmax=0.15)
ax.set_xticks(range(len(delta_data.columns)))
ax.set_xticklabels(
    [TTA_LABELS.get(m, m) for m in delta_data.columns],
    rotation=50,
    ha="right",
    fontsize=FONTSIZE - 1,
)
ax.set_yticks(range(len(delta_data.index)))
ax.set_yticklabels([f"{d}\n{m}" for d, m in delta_data.index], fontsize=FONTSIZE - 1)
ax.set_title(
    f"Δ Bal. Acc. (vs {baseline_method}) per-dataset/model",
    fontsize=FONTSIZE,
    weight="bold",
)

for i in range(len(delta_data.index)):
    for j in range(len(delta_data.columns)):
        val = delta_data.iloc[i, j]
        p = pval_data.iloc[i, j]
        stars = get_stars(p)
        color = "white" if abs(val) > 0.07 else "black"
        ax.text(
            j,
            i,
            f"{val:+.2f}{stars}",
            ha="center",
            va="center",
            fontsize=FONTSIZE - 2,
            color=color,
        )

plt.tight_layout()
plt.savefig("fig_tta_comparison_combined.pdf", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: fig_tta_comparison_combined.pdf")